**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Intro to Convolutional Neural Networks

A CNN is [DSP](../../Intro_DSP/README.md) that trains itself: convolution — the operation you mastered in [Foundations of Signal Processing](../../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb) — with the filter taps turned into **learnable weights**. We classify signals by their spectrograms and then *open the hood* to see what filters the network chose to learn.

## 0. Introduction

Why not use the MLP from the [ANN workshop](../Intro_ANN/Intro_ANN.ipynb) on images/spectrograms? Because a dense layer on a 128×128 input needs millions of weights and must re-learn the same edge detector at every location. Convolution fixes both with two priors:

- **Locality** — nearby pixels/samples are related; a small kernel suffices.
- **Weight sharing** — a feature is the same feature *wherever* it appears, so one kernel slides everywhere (this is what makes the layer a convolution!).

## 1. Pre-requisites

- [Intro to ANN](../Intro_ANN/Intro_ANN.ipynb) — layers, backprop, the training loop.
- [Intro to PyTorch](../../Intro_DL_4_Physics/intro_pytorch/intro_pytorch.ipynb) — `nn.Module`, Dataloaders.
- [Foundations of Signal Processing](../../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb) Sessions 4–6 — convolution & the STFT/spectrogram.

In [1]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from scipy import signal as sig

torch.manual_seed(0)
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 2 — *Convolution as a Learned Filter Bank* (~35 min)
**Goal:** map conv/pool/stride/receptive-field onto DSP concepts you already own.
**Builds on:** [ANN](../Intro_ANN/Intro_ANN.ipynb); [DSP Foundations](../../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb). &nbsp; **Feeds into:** Session 2 (train & inspect).

---

## 2. Theory: the CNN Vocabulary, Translated from DSP

| CNN term | DSP translation |
|---|---|
| kernel / filter | FIR filter taps (2-D), **learned** |
| feature map | the filtered output signal |
| channels | a *filter bank* — many filters in parallel |
| stride | decimation / downsampling |
| pooling | nonlinear downsampling (max = "was the feature present anywhere here?") |
| receptive field | the region of input a deep unit can "see" — grows with depth |

💡 **Intuition.** A convolutional *layer* is a filter bank followed by a nonlinearity. Stacking layers composes filters into detectors of ever larger, ever more abstract patterns: taps → edges → textures → shapes. The network is a *hierarchical* filter bank whose every tap was chosen by gradient descent instead of by [Filter Design](../../Intro_DSP/Filter_Design.ipynb).

### 2.1. Seeing One Convolution

Before trusting `nn.Conv2d`, apply a hand-made kernel: a vertical-edge detector is just a 2-D FIR high-pass in one direction.

In [2]:
# A test image: a bright square on darkness
img = np.zeros((64, 64), dtype=np.float32)
img[16:48, 20:44] = 1.0

k_edge = np.array([[1, 0, -1]], dtype=np.float32)          # d/dx-ish kernel
edges = sig.convolve2d(img, k_edge, mode="same")

fig, axes = plt.subplots(1, 2, figsize=(7, 3))
axes[0].imshow(img, cmap="gray"); axes[0].set_title("input")
axes[1].imshow(edges, cmap="RdBu"); axes[1].set_title("after [1, 0, −1]: vertical edges")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

/tmp/ipykernel_1849661/2189497216.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


`nn.Conv2d(1, 16, 3)` learns **16 such 3×3 kernels at once** — plus their biases — and backprop nudges every tap, exactly as in the [ANN workshop](../Intro_ANN/Intro_ANN.ipynb), just with shared weights.

---
### 🕐 Session 2 of 2 — *Train a CNN on Spectrograms* (~40 min)
**Goal:** classify chirps vs tones vs noise bursts from their spectrograms; visualize learned kernels.
**Builds on:** Session 1.

---

## 3. Application: What Kind of Signal Is This?

A task straight from a signals lab: given a spectrogram, is the emitter a **rising chirp**, a **pure tone**, or a **noise burst**? (The same pipeline classifies radar pulses, bird calls, and modulation schemes.)

In [3]:
fs, T = 1024, 1.0
t = np.arange(0, T, 1 / fs)

def make_example(cls):
    if cls == 0:    # chirp
        f0 = rng.uniform(50, 150)
        x = sig.chirp(t, f0=f0, f1=f0 + rng.uniform(100, 250), t1=T)
    elif cls == 1:  # tone
        x = np.sin(2 * np.pi * rng.uniform(80, 350) * t)
    else:           # noise burst
        x = np.zeros_like(t)
        s = rng.integers(0, len(t) // 2)
        x[s:s + len(t) // 3] = rng.standard_normal(len(t) // 3)
    x = x + 0.3 * rng.standard_normal(len(t))
    _, _, S = sig.stft(x, fs=fs, nperseg=64)
    S = np.log1p(np.abs(S))[:32, :32]                     # 32×32 log-spectrogram
    return (S - S.mean()) / (S.std() + 1e-6)

X = np.stack([make_example(c % 3) for c in range(900)]).astype(np.float32)
y = np.array([c % 3 for c in range(900)])

fig, axes = plt.subplots(1, 3, figsize=(8, 2.6))
for ax, c, name in zip(axes, [0, 1, 2], ["chirp", "tone", "noise burst"]):
    ax.imshow(X[c], origin="lower", aspect="auto")
    ax.set_title(name); ax.axis("off")
plt.tight_layout(); plt.show()

/tmp/ipykernel_1849661/940485882.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [4]:
class SpecCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 32→16
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 16→8
        )
        self.classify = nn.Sequential(
            nn.Flatten(), nn.Linear(32 * 8 * 8, 64), nn.ReLU(), nn.Linear(64, 3))

    def forward(self, x):
        return self.classify(self.features(x))

model = SpecCNN()
print(sum(p.numel() for p in model.parameters()), "parameters",
      "(an MLP flattening 32×32 to 1024→512 would already need ~525k in its first layer)")

136131 parameters (an MLP flattening 32×32 to 1024→512 would already need ~525k in its first layer)


In [5]:
Xt = torch.from_numpy(X).unsqueeze(1)          # (N, 1, 32, 32)
yt = torch.from_numpy(y)
idx = torch.randperm(len(Xt))
tr, te = idx[:700], idx[700:]

loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(Xt[tr], yt[tr]), batch_size=64, shuffle=True)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(6):
    model.train(); total = 0.0
    for xb, yb in loader:
        opt.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward(); opt.step()
        total += loss.item() * len(xb)
    model.eval()
    with torch.no_grad():
        acc = (model(Xt[te]).argmax(1) == yt[te]).float().mean()
    print(f"epoch {epoch}: loss {total / len(tr):.3f}   test acc {acc:.1%}")

epoch 0: loss 0.660   test acc 95.0%
epoch 1: loss 0.089   test acc 100.0%
epoch 2: loss 0.007   test acc 100.0%
epoch 3: loss 0.001   test acc 100.0%
epoch 4: loss 0.000   test acc 100.0%
epoch 5: loss 0.000   test acc 100.0%


### 3.1. Open the Hood: What Did It Learn?

First-layer kernels are directly plottable — and on spectrogram input, oriented kernels are *frequency-trajectory detectors*: a tilted edge detector fires on chirps, a horizontal one on steady tones.

In [6]:
kernels = model.features[0].weight.detach().squeeze(1)   # (16, 3, 3)
fig, axes = plt.subplots(2, 8, figsize=(9, 2.6))
for ax, k in zip(axes.ravel(), kernels):
    ax.imshow(k, cmap="RdBu"); ax.axis("off")
plt.suptitle("The 16 learned 3×3 kernels — a filter bank nobody designed")
plt.tight_layout(); plt.show()

/tmp/ipykernel_1849661/4163168610.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [7]:
# And the feature maps for one chirp: which filters fire, and where?
with torch.no_grad():
    fmap = model.features[0](Xt[0:1]).squeeze(0)
fig, axes = plt.subplots(2, 8, figsize=(9, 2.8))
for ax, f in zip(axes.ravel(), fmap):
    ax.imshow(f, origin="lower", aspect="auto"); ax.axis("off")
plt.suptitle("Feature maps of a chirp after layer 1")
plt.tight_layout(); plt.show()

/tmp/ipykernel_1849661/200881441.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 4. Conclusion

A CNN = hierarchical learned filter bank: locality and weight sharing shrink millions of weights to thousands, pooling buys translation tolerance, and the learned kernels are *readable* — on spectrograms they recover the matched filters a signal engineer would have designed.

---
## Where next

- [Intro to Transformers](../../Intro_DL_4_Physics/intro_transformers/intro_transformers.ipynb) — attention removes even the locality prior and lets the data decide connectivity.
- [Filter Design](../../Intro_DSP/Filter_Design.ipynb) — the hand-designed baseline these kernels replace.
- [Scaling Neural Networks](../README.md#workshop-3--scaling-neural-networks-available) — this architecture, three orders of magnitude larger.